# Experiment: Preparación de AudioSet-EV

**Pregunta.** ¿Cómo se limpia, parte (splits oficiales + val por `yt_id`), normaliza y transforma el corpus de train sin fugas ni mezclar sireNNet?

**Criterio de éxito.** Índice limpio, `audioset_ev_splits.csv` con `yt_id` disjuntos, scaler ajustado solo en train, y una muestra de features/log-mel. No se entrena un clasificador.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

SEED = 7

# Local: data/raw/<corpus>
# Kaggle (Add Input clásico): /kaggle/input/<slug>
# Kaggle (datasets/user): /kaggle/input/datasets/<user>/<slug>/<slug>
IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_ROOT = Path("/kaggle/input")
    FIGURES_DIR = Path("/kaggle/working/reports/figures")
    TABLES_DIR = Path("/kaggle/working/reports/tables")
else:
    here = Path.cwd().resolve()
    REPO_ROOT = here if (here / "data" / "raw").exists() else here.parent
    DATA_ROOT = REPO_ROOT / "data" / "raw"
    FIGURES_DIR = REPO_ROOT / "reports" / "figures"
    TABLES_DIR = REPO_ROOT / "reports" / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_SLUGS = {
    "sirennet": ("sirennet",),
    "lssiren": ("lssiren",),
    "urbansound8k": ("urbansound8k",),
    "audioset_ev": ("audioset_ev", "audioset-ev"),
}


def is_kaggle() -> bool:
    return IS_KAGGLE


def figures_dir() -> Path:
    return FIGURES_DIR


def tables_dir() -> Path:
    return TABLES_DIR


def resolve_corpus(name: str) -> Path | None:
    candidates: list[Path] = []
    for slug in CORPUS_SLUGS[name]:
        candidates.append(DATA_ROOT / slug)
        datasets = DATA_ROOT / "datasets"
        if datasets.exists():
            for user_dir in datasets.iterdir():
                if not user_dir.is_dir():
                    continue
                candidates.append(user_dir / slug)
                candidates.append(user_dir / slug / slug)
    existing = [path for path in candidates if path.exists()]
    if not existing:
        return None
    return max(existing, key=lambda path: len(path.parts))


@dataclass(frozen=True)
class CorpusPaths:
    sirennet: Path | None
    lssiren: Path | None
    urbansound8k: Path | None
    audioset_ev: Path | None

    def available(self) -> dict[str, Path]:
        found = {
            "sirennet": self.sirennet,
            "lssiren": self.lssiren,
            "urbansound8k": self.urbansound8k,
            "audioset_ev": self.audioset_ev,
        }
        return {key: path for key, path in found.items() if path is not None}


def corpus_paths() -> CorpusPaths:
    return CorpusPaths(
        sirennet=resolve_corpus("sirennet"),
        lssiren=resolve_corpus("lssiren"),
        urbansound8k=resolve_corpus("urbansound8k"),
        audioset_ev=resolve_corpus("audioset_ev"),
    )


print("kaggle:", IS_KAGGLE)
print("DATA_ROOT:", DATA_ROOT)
print("FIGURES_DIR:", FIGURES_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("available:", list(corpus_paths().available()))
SEED


kaggle: False
DATA_ROOT: /home/jeancdevx/dev/doppler/doppler-ml/data/raw
FIGURES_DIR: /home/jeancdevx/dev/doppler/doppler-ml/reports/figures
TABLES_DIR: /home/jeancdevx/dev/doppler/doppler-ml/reports/tables
available: ['sirennet', 'lssiren', 'urbansound8k', 'audioset_ev']


7

In [2]:
# Inventario de archivos
"""Inventario de archivos de audio del corpus Doppler."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

SIRENNET_CLASS_MAP = {
    "ambulance": "ambulance",
    "police": "police",
    "firetruck": "firetruck",
    "fire_truck": "firetruck",
    "fire": "firetruck",
    "traffic": "traffic",
}

LSSIREN_POSITIVE_HINTS = ("emergency", "siren", "ambulance")
LSSIREN_NEGATIVE_HINTS = ("road", "noise", "traffic")
AUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}


def is_audio(path: Path) -> bool:
    return path.suffix.lower() in AUDIO_SUFFIXES


def infer_sirennet_label(path: Path) -> str | None:
    parts = [p.lower() for p in path.parts]
    stem = path.stem.lower()
    for key, label in SIRENNET_CLASS_MAP.items():
        if key in parts or stem.startswith(key) or f"_{key}_" in f"_{stem}_":
            return label
    return None


def infer_lssiren_label(path: Path) -> str | None:
    blob = " ".join(p.lower() for p in path.parts)
    if any(h in blob for h in LSSIREN_POSITIVE_HINTS) and "road" not in blob:
        return "siren"
    if any(h in blob for h in LSSIREN_NEGATIVE_HINTS):
        return "road_noise"
    return None


def scan_sirennet(root: Path) -> pd.DataFrame:
    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        rows.append(
            {
                "corpus": "sirennet",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": infer_sirennet_label(path) or "unknown",
                "task": "multiclass",
            }
        )
    return pd.DataFrame(rows)


LSSIREN_CSV_COLUMNS = [
    "filename",
    "chroma_stft",
    "rmse",
    "spectral_centroid",
    "spectral_bandwidth",
    "rolloff",
    "zero_crossing_rate",
    *[f"mfcc{i}" for i in range(1, 21)],
    "label",
]


def read_lssiren_features(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    if "filename" in raw.columns:
        return raw
    return pd.read_csv(path, header=None, names=LSSIREN_CSV_COLUMNS)


def scan_lssiren(root: Path) -> pd.DataFrame:
    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        label = infer_lssiren_label(path) or "unknown"
        rows.append(
            {
                "corpus": "lssiren",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": label,
                "task": "binary",
            }
        )
    if rows:
        return pd.DataFrame(rows)

    for csv_path in root.glob("*.csv"):
        feat = read_lssiren_features(csv_path)
        label_col = "label" if "label" in feat.columns else feat.columns[-1]
        name_col = "filename" if "filename" in feat.columns else feat.columns[0]
        for _, row in feat.iterrows():
            raw_label = str(row[label_col]).strip().lower()
            label = "siren" if raw_label in {"ambulance", "siren", "emergency"} else "road_noise"
            rows.append(
                {
                    "corpus": "lssiren",
                    "path": str(csv_path.parent / str(row[name_col])),
                    "relpath": str(row[name_col]),
                    "label": label,
                    "task": "binary",
                    "source": "feature_csv",
                }
            )
    return pd.DataFrame(rows)


def scan_urbansound8k(root: Path) -> pd.DataFrame:
    csv_candidates = list(root.rglob("UrbanSound8K.csv"))
    if csv_candidates:
        meta = pd.read_csv(csv_candidates[0])
        if "class" not in meta.columns and "class_name" in meta.columns:
            meta = meta.rename(columns={"class_name": "class"})
        audio_root = csv_candidates[0].parent.parent / "audio"
        if not audio_root.exists():
            audio_root = root / "audio"
            if not audio_root.exists():
                audio_root = root
        rows = []
        for _, row in meta.iterrows():
            fold = int(row["fold"])
            fname = row["slice_file_name"]
            path = audio_root / f"fold{fold}" / fname
            rows.append(
                {
                    "corpus": "urbansound8k",
                    "path": str(path),
                    "relpath": f"fold{fold}/{fname}",
                    "label": row["class"],
                    "task": "urban_scene",
                    "fold": fold,
                    "fsID": row.get("fsID"),
                    "classID": row.get("classID"),
                    "salience": row.get("salience"),
                }
            )
        return pd.DataFrame(rows)

    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        rows.append(
            {
                "corpus": "urbansound8k",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": path.parent.name,
                "task": "urban_scene",
            }
        )
    return pd.DataFrame(rows)


AUDIOSET_EV_MIDS = {
    "/m/04qvtq": "police",
    "/m/012n7d": "ambulance",
    "/m/012ndj": "firetruck",
}
AUDIOSET_EV_GENERIC_MIDS = {
    "/m/03j1ly",  # Emergency vehicle
    "/m/03kmc9",  # Siren
}


def parse_audioset_mids(raw) -> list[str]:
    if raw is None:
        return []
    try:
        if pd.isna(raw):
            return []
    except (TypeError, ValueError):
        pass
    text = str(raw).strip()
    if not text or text.lower() in {"nan", "none"}:
        return []
    found = []
    for token in (
        text.replace("[", " ")
        .replace("]", " ")
        .replace("'", " ")
        .replace('"', " ")
        .replace(",", " ")
        .split()
    ):
        if token.startswith("/m/") or token.startswith("/g/"):
            found.append(token)
    return found


def map_audioset_labels(mids: list[str]) -> tuple[str, bool, str]:
    mapped: list[str] = []
    for mid in mids:
        label = AUDIOSET_EV_MIDS.get(mid)
        if label and label not in mapped:
            mapped.append(label)
    multi = len(mapped) > 1
    if not mapped:
        if any(mid in AUDIOSET_EV_GENERIC_MIDS for mid in mids):
            type_label = "siren_untyped"
        else:
            type_label = "unknown"
    elif multi:
        type_label = "multi"
    else:
        type_label = mapped[0]
    return type_label, multi, "|".join(mapped)


def _yt_id_from_stem(stem: str) -> str:
    if stem.startswith("Y") and len(stem) > 1:
        return stem[1:]
    return stem


def _audioset_segment_from_path(path: Path) -> str | None:
    parts = [p.lower() for p in path.parts]
    for key in ("unbalanced", "balanced_train", "eval"):
        if key in parts:
            return key
    return None


def _audioset_polarity_from_path(path: Path) -> str | None:
    blob = "/".join(p.lower() for p in path.parts)
    if "negative_files" in blob or "/negatives/" in blob:
        return "negative"
    if "positive_files" in blob or "/positives/" in blob:
        return "positive"
    return None


def _truthy_downloaded(val) -> bool:
    if val is True:
        return True
    if val is False or val is None:
        return False
    try:
        if pd.isna(val):
            return False
    except (TypeError, ValueError):
        pass
    return str(val).strip().lower() in {"true", "1", "yes"}


def _read_audioset_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    return df


def scan_audioset_ev(root: Path) -> pd.DataFrame:
    pos_csv = None
    neg_csv = None
    for csv_path in root.rglob("*.csv"):
        name = csv_path.name.lower()
        if name == "ev_positives.csv":
            pos_csv = csv_path
        elif name == "ev_negatives.csv":
            neg_csv = csv_path

    meta_rows = []
    if pos_csv is not None:
        pos = _read_audioset_csv(pos_csv)
        if "downloaded" in pos.columns:
            pos = pos[pos["downloaded"].map(_truthy_downloaded)]
        for _, row in pos.iterrows():
            yt_id = str(row.get("yt_id", row.iloc[0])).strip()
            mids = parse_audioset_mids(row.get("positive_labels", row.get("labels")))
            type_label, multi, joined = map_audioset_labels(mids)
            meta_rows.append(
                {
                    "yt_id": yt_id,
                    "polarity": "positive",
                    "segment_type": str(row.get("segment_type", "")).strip() or None,
                    "downloaded_flag": row.get("downloaded"),
                    "type_label": type_label,
                    "multi_positive": multi,
                    "ev_labels": joined,
                    "mids": "|".join(mids),
                }
            )
    if neg_csv is not None:
        neg = _read_audioset_csv(neg_csv)
        if "downloaded" in neg.columns:
            neg = neg[neg["downloaded"].map(_truthy_downloaded)]
        for _, row in neg.iterrows():
            yt_id = str(row.get("yt_id", row.iloc[0])).strip()
            meta_rows.append(
                {
                    "yt_id": yt_id,
                    "polarity": "negative",
                    "segment_type": str(row.get("segment_type", "")).strip() or None,
                    "downloaded_flag": row.get("downloaded"),
                    "type_label": "urban_negative",
                    "multi_positive": False,
                    "ev_labels": "",
                    "mids": "",
                }
            )
    meta = pd.DataFrame(meta_rows)
    meta_by_id = {}
    if not meta.empty:
        meta_by_id = {str(r["yt_id"]): r for r in meta.to_dict(orient="records")}

    wav_rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        yt_id = _yt_id_from_stem(path.stem)
        info = meta_by_id.get(yt_id, {})
        polarity = info.get("polarity") or _audioset_polarity_from_path(path) or "unknown"
        segment = info.get("segment_type") or _audioset_segment_from_path(path) or "unknown"
        if polarity == "negative":
            type_label = "urban_negative"
            multi = False
            ev_labels = ""
            task = "binary"
        else:
            type_label = info.get("type_label") or "unknown"
            multi = bool(info.get("multi_positive", False))
            ev_labels = info.get("ev_labels") or ""
            task = "multiclass"
        wav_rows.append(
            {
                "corpus": "audioset_ev",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": type_label,
                "task": task,
                "yt_id": yt_id,
                "group_id": yt_id,
                "polarity": polarity,
                "segment_type": segment,
                "multi_positive": multi,
                "ev_labels": ev_labels,
                "has_wav": True,
            }
        )

    wav_df = pd.DataFrame(wav_rows)
    if meta.empty:
        return wav_df

    seen = set(wav_df["yt_id"].astype(str)) if not wav_df.empty else set()
    missing_rows = []
    for rec in meta.to_dict(orient="records"):
        if str(rec["yt_id"]) in seen:
            continue
        polarity = rec["polarity"]
        missing_rows.append(
            {
                "corpus": "audioset_ev",
                "path": "",
                "relpath": "",
                "label": rec["type_label"],
                "task": "binary" if polarity == "negative" else "multiclass",
                "yt_id": rec["yt_id"],
                "group_id": rec["yt_id"],
                "polarity": polarity,
                "segment_type": rec["segment_type"] or "unknown",
                "multi_positive": rec["multi_positive"],
                "ev_labels": rec["ev_labels"],
                "has_wav": False,
            }
        )
    if missing_rows:
        wav_df = pd.concat([wav_df, pd.DataFrame(missing_rows)], ignore_index=True)
    return wav_df


def assign_audioset_protocol_split(polarity: str, segment_type: str) -> str:
    pol = (polarity or "").strip().lower()
    seg = (segment_type or "").strip().lower()
    if seg == "eval":
        return "test"
    if pol == "positive" and seg in {"unbalanced", "unbalanced_train"}:
        return "train_pool"
    if pol == "negative" and seg == "balanced_train":
        return "train_pool"
    if pol == "positive" and seg == "balanced_train":
        return "balanced_train_ref"
    return "unused"


In [3]:
# Lectura de audio y descriptores
"""Lectura de audio y características con librosa, o scipy si no está disponible."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

try:
    import librosa

    HAS_LIBROSA = True
except Exception:  # pragma: no cover - entorno sin ruedas de librosa
    HAS_LIBROSA = False
    librosa = None  # type: ignore

from scipy.io import wavfile
from scipy.signal import get_window, spectrogram, stft

try:
    import soundfile as sf

    HAS_SOUNDFILE = True
except Exception:
    HAS_SOUNDFILE = False
    sf = None  # type: ignore


TARGET_SR = 22050
AUDIOSET_DURATION_S = 10.0


@dataclass
class AudioClip:
    y: np.ndarray
    sr: int


def peak_normalize(y: np.ndarray, peak: float = 0.99) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    max_abs = float(np.max(np.abs(y))) if y.size else 0.0
    if max_abs < 1e-8:
        return y
    return (y / max_abs) * peak


def rms_normalize(y: np.ndarray, target_rms: float = 0.1) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    rms = float(np.sqrt(np.mean(np.square(y)))) if y.size else 0.0
    if rms < 1e-8:
        return y
    return y * (target_rms / rms)


def pad_or_trim(y: np.ndarray, sr: int, duration_s: float) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    n = int(round(duration_s * sr))
    if n <= 0:
        return y
    if len(y) >= n:
        return y[:n]
    out = np.zeros(n, dtype=np.float32)
    out[: len(y)] = y
    return out


def load_audio(path: str, sr: int = TARGET_SR, duration: float | None = None) -> AudioClip:
    if HAS_LIBROSA:
        y, out_sr = librosa.load(path, sr=sr, mono=True, duration=duration)
        return AudioClip(y=np.asarray(y, dtype=np.float32), sr=out_sr)

    if HAS_SOUNDFILE:
        y, file_sr = sf.read(path, always_2d=False)
        y = np.asarray(y, dtype=np.float32)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if duration is not None:
            y = y[: int(duration * file_sr)]
        if sr and file_sr != sr:
            duration_s = len(y) / float(file_sr)
            n_out = max(1, int(duration_s * sr))
            x_old = np.linspace(0.0, duration_s, num=len(y), endpoint=False)
            x_new = np.linspace(0.0, duration_s, num=n_out, endpoint=False)
            y = np.interp(x_new, x_old, y).astype(np.float32)
            file_sr = sr
        return AudioClip(y=y, sr=int(file_sr))

    file_sr, data = wavfile.read(path)
    y = np.asarray(data, dtype=np.float32)
    if y.ndim > 1:
        y = y.mean(axis=1)
    max_abs = np.max(np.abs(y)) or 1.0
    if max_abs > 1.5:
        y = y / 32768.0
    if duration is not None:
        y = y[: int(duration * file_sr)]
    if sr and file_sr != sr:
        duration_s = len(y) / file_sr
        n_out = int(duration_s * sr)
        x_old = np.linspace(0.0, duration_s, num=len(y), endpoint=False)
        x_new = np.linspace(0.0, duration_s, num=n_out, endpoint=False)
        y = np.interp(x_new, x_old, y).astype(np.float32)
        file_sr = sr
    return AudioClip(y=y, sr=file_sr)


def duration_seconds(path: str) -> float:
    return float(wav_probe(path)["duration_s"])


def wav_probe(path: str) -> dict:
    """Metadatos baratos sin resamplear."""
    if HAS_SOUNDFILE:
        info = sf.info(path)
        return {
            "sr": int(info.samplerate),
            "n_channels": int(info.channels),
            "n_samples": int(info.frames),
            "duration_s": float(info.duration),
            "dtype": str(info.subtype),
        }
    try:
        sr, data = wavfile.read(path)
        n_channels = 1 if np.asarray(data).ndim == 1 else np.asarray(data).shape[1]
        n_samples = int(np.asarray(data).shape[0])
        return {
            "sr": int(sr),
            "n_channels": int(n_channels),
            "n_samples": n_samples,
            "duration_s": n_samples / float(sr),
            "dtype": str(np.asarray(data).dtype),
        }
    except Exception:
        if HAS_LIBROSA:
            y, sr = librosa.load(path, sr=None, mono=False)
            y = np.asarray(y)
            n_channels = 1 if y.ndim == 1 else y.shape[0]
            n_samples = y.shape[-1]
            return {
                "sr": int(sr),
                "n_channels": int(n_channels),
                "n_samples": int(n_samples),
                "duration_s": n_samples / float(sr),
                "dtype": str(y.dtype),
            }
        raise


def log_mel_spectrogram(clip: AudioClip, n_mels: int = 64, n_fft: int = 1024, hop: int = 256) -> np.ndarray:
    if HAS_LIBROSA:
        S = librosa.feature.melspectrogram(y=clip.y, sr=clip.sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop)
        return librosa.power_to_db(S, ref=np.max)
    f, t, Sxx = spectrogram(clip.y, fs=clip.sr, nperseg=n_fft, noverlap=n_fft - hop, window="hann")
    # Aproximación: filtro triangular en Hz de Mel.
    mel_f = _hz_to_mel(f)
    edges = np.linspace(mel_f.min(), mel_f.max(), n_mels + 2)
    mels = np.zeros((n_mels, Sxx.shape[1]), dtype=np.float32)
    for i in range(n_mels):
        lo, mid, hi = edges[i], edges[i + 1], edges[i + 2]
        w = np.zeros_like(mel_f)
        left = np.logical_and(mel_f >= lo, mel_f <= mid)
        right = np.logical_and(mel_f >= mid, mel_f <= hi)
        if np.any(left):
            w[left] = (mel_f[left] - lo) / max(mid - lo, 1e-8)
        if np.any(right):
            w[right] = (hi - mel_f[right]) / max(hi - mid, 1e-8)
        mels[i] = w @ Sxx
    mels = np.maximum(mels, 1e-10)
    return 10.0 * np.log10(mels / np.max(mels))


def mfcc(clip: AudioClip, n_mfcc: int = 13) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.mfcc(y=clip.y, sr=clip.sr, n_mfcc=n_mfcc)
    log_mel = log_mel_spectrogram(clip, n_mels=40)
    # DCT tipo II sobre el eje mel.
    n_mels, n_frames = log_mel.shape
    n = np.arange(n_mels)
    k = np.arange(n_mfcc)[:, None]
    dct = np.cos(np.pi * k * (2 * n + 1) / (2.0 * n_mels))
    return dct @ log_mel


def spectral_centroid(clip: AudioClip) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.spectral_centroid(y=clip.y, sr=clip.sr)[0]
    f, _, Zxx = stft(clip.y, fs=clip.sr, nperseg=1024)
    mag = np.abs(Zxx)
    denom = np.sum(mag, axis=0) + 1e-10
    return (f[:, None] * mag).sum(axis=0) / denom


def spectral_bandwidth(clip: AudioClip) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.spectral_bandwidth(y=clip.y, sr=clip.sr)[0]
    f, _, Zxx = stft(clip.y, fs=clip.sr, nperseg=1024)
    mag = np.abs(Zxx)
    denom = np.sum(mag, axis=0) + 1e-10
    centroid = (f[:, None] * mag).sum(axis=0) / denom
    var = ((f[:, None] - centroid) ** 2 * mag).sum(axis=0) / denom
    return np.sqrt(var)


def spectral_rolloff(clip: AudioClip, roll_percent: float = 0.85) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.spectral_rolloff(y=clip.y, sr=clip.sr, roll_percent=roll_percent)[0]
    f, _, Zxx = stft(clip.y, fs=clip.sr, nperseg=1024)
    mag = np.abs(Zxx)
    csum = np.cumsum(mag, axis=0)
    thresh = roll_percent * (csum[-1] + 1e-10)
    idx = np.argmax(csum >= thresh, axis=0)
    return f[idx]


def zero_crossing_rate(clip: AudioClip, frame_length: int = 2048, hop: int = 512) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.zero_crossing_rate(clip.y, frame_length=frame_length, hop_length=hop)[0]
    y = clip.y
    n = 1 + max(0, (len(y) - frame_length) // hop)
    out = np.zeros(n, dtype=np.float32)
    for i in range(n):
        frame = y[i * hop : i * hop + frame_length]
        out[i] = np.mean(np.abs(np.diff(np.signbit(frame))))
    return out


def _hz_to_mel(hz: np.ndarray) -> np.ndarray:
    return 2595.0 * np.log10(1.0 + hz / 700.0)


def stft_db(clip: AudioClip, n_fft: int = 1024, hop: int = 256) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if HAS_LIBROSA:
        S = np.abs(librosa.stft(clip.y, n_fft=n_fft, hop_length=hop))
        db = librosa.amplitude_to_db(S, ref=np.max)
        freqs = librosa.fft_frequencies(sr=clip.sr, n_fft=n_fft)
        times = librosa.frames_to_time(np.arange(db.shape[1]), sr=clip.sr, hop_length=hop)
        return freqs, times, db
    window = get_window("hann", n_fft)
    f, t, Zxx = stft(clip.y, fs=clip.sr, window=window, nperseg=n_fft, noverlap=n_fft - hop)
    mag = np.abs(Zxx)
    db = 20.0 * np.log10(np.maximum(mag, 1e-10) / (np.max(mag) + 1e-10))
    return f, t, db


## Plan

- 4.1 limpieza del índice (CSV ↔ WAV, flags).
- 4.5 splits oficiales **antes** de ajustar el scaler.
- 4.2–4.4: peak/RMS por clip, resample 22 050 Hz, pad/trim 10 s, log-mel, MFCC+descriptores en muestra.
- No 70/15/15 aleatorio. No mezclar corpora. No denoise.


In [4]:
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path as _P

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
FIG = figures_dir()
TAB = tables_dir()
print("librosa:", HAS_LIBROSA)

paths = corpus_paths()
if paths.audioset_ev is None:
    raise FileNotFoundError("Monta AudioSet-EV en data/raw/audioset_ev o como Input de Kaggle (audioset-ev).")
ev = scan_audioset_ev(paths.audioset_ev)
print(ev.groupby(["polarity", "segment_type"]).size() if not ev.empty else "empty")
ev.head()


librosa: False


polarity  segment_type    
negative  balanced_train      10963
          eval                 9953
positive  balanced_train        124
          eval                  127
          unbalanced_train     7073
dtype: int64


,corpus,path,relpath,label,task,yt_id,group_id,polarity,segment_type,multi_positive,ev_labels,has_wav
0,audioset_ev,,,firetruck,multiclass,-RBs9pPhHY8,-RBs9pPhHY8,positive,eval,False,firetruck,False
1,audioset_ev,,,ambulance,multiclass,02Ak1eIyj3M,02Ak1eIyj3M,positive,eval,False,ambulance,False
2,audioset_ev,,,ambulance,multiclass,03Q2SbeP_cw,03Q2SbeP_cw,positive,eval,False,ambulance,False
3,audioset_ev,,,police,multiclass,0N0C0Wbe6AI,0N0C0Wbe6AI,positive,eval,False,police,False
4,audioset_ev,,,police,multiclass,1F9zCsJyw6k,1F9zCsJyw6k,positive,eval,False,police,False


## 4.1 Limpieza de datos

No se reescriben WAV crudos. Se descartan ilegibles y se marcan clipping/duración anómala en los archivos presentes. `group_id = yt_id`.


In [5]:
clean = ev.copy()
clean["group_id"] = clean["yt_id"].astype(str)
clean["detection_label"] = np.where(clean["polarity"] == "positive", "siren", "urban_negative")
clean["type_eligible"] = (
    (clean["polarity"] == "positive")
    & (~clean["multi_positive"].fillna(False).astype(bool))
    & (clean["label"].isin(["ambulance", "police", "firetruck"]))
)
clean["keep"] = True
clean["flag_missing_wav"] = ~clean["has_wav"].fillna(False).astype(bool)
clean["flag_unreadable"] = False
clean["flag_clipping"] = False
clean["flag_short"] = False
clean["duration_s"] = np.nan
clean["peak"] = np.nan

wav_mask = clean["path"].map(lambda p: bool(p) and _P(str(p)).exists())
for idx in clean.index[wav_mask]:
    path = clean.at[idx, "path"]
    try:
        meta = wav_probe(path)
        dur = float(meta["duration_s"]) if meta.get("duration_s") is not None else np.nan
        clean.at[idx, "duration_s"] = dur
        if dur == dur and dur < 1.0:
            clean.at[idx, "flag_short"] = True
        clip = load_audio(path, duration=min(10.0, dur) if dur == dur else 10.0)
        peak = float(np.max(np.abs(clip.y))) if clip.y.size else 0.0
        clean.at[idx, "peak"] = peak
        if peak >= 0.999:
            clean.at[idx, "flag_clipping"] = True
    except Exception:
        clean.at[idx, "flag_unreadable"] = True
        clean.at[idx, "keep"] = False

clean.loc[clean["flag_missing_wav"] & (clean.get("downloaded_flag", True) == False), "keep"] = False
clean.to_csv(TAB / "audioset_ev_clean_index.csv", index=False)
flag_summary = pd.DataFrame({
    "n": [len(clean)],
    "keep": [int(clean["keep"].sum())],
    "missing_wav": [int(clean["flag_missing_wav"].sum())],
    "unreadable": [int(clean["flag_unreadable"].sum())],
    "clipping": [int(clean["flag_clipping"].sum())],
    "short": [int(clean["flag_short"].sum())],
    "multi_positive": [int(clean["multi_positive"].fillna(False).astype(bool).sum())],
})
flag_summary.to_csv(TAB / "audioset_ev_clean_summary.csv", index=False)
flag_summary


,n,keep,missing_wav,unreadable,clipping,short,multi_positive
0,28240,28240,28240,0,0,0,829


## 4.5 División del conjunto (antes del scaler)

Train = positivos `unbalanced` + negativos `balanced_train`. Test = `eval` oficial. Val = 15 % de grupos `yt_id` del train pool (`SEED=7`). `balanced_train` positivo se documenta y no sustituye a `unbalanced`.


In [6]:
splits = clean.copy()
splits["protocol_pool"] = [
    assign_audioset_protocol_split(p, s) for p, s in zip(splits["polarity"], splits["segment_type"])
]
splits["split"] = splits["protocol_pool"].replace({"train_pool": "train", "balanced_train_ref": "balanced_train_ref"})

train_pool = splits[splits["protocol_pool"] == "train_pool"].copy()
if train_pool.empty:
    print("train_pool vacío")
    val_ids = set()
else:
    groups = train_pool["group_id"].astype(str).to_numpy()
    dummy = np.zeros(len(train_pool))
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
    train_idx, val_idx = next(gss.split(dummy, groups=groups))
    val_ids = set(train_pool.iloc[val_idx]["group_id"].astype(str))
    is_val = splits["group_id"].astype(str).isin(val_ids) & (splits["protocol_pool"] == "train_pool")
    splits.loc[is_val, "split"] = "val"

overlap = {
    "train_val": len(set(splits.loc[splits["split"] == "train", "group_id"]) & set(splits.loc[splits["split"] == "val", "group_id"])),
    "train_test": len(set(splits.loc[splits["split"] == "train", "group_id"]) & set(splits.loc[splits["split"] == "test", "group_id"])),
    "val_test": len(set(splits.loc[splits["split"] == "val", "group_id"]) & set(splits.loc[splits["split"] == "test", "group_id"])),
}
print("yt_id overlap (debe ser 0):", overlap)

split_tbl = splits.groupby(["split", "detection_label"]).size().reset_index(name="n")
split_tbl.to_csv(TAB / "audioset_ev_split_counts.csv", index=False)
splits.to_csv(TAB / "audioset_ev_splits.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=split_tbl, x="split", y="n", hue="detection_label", ax=ax)
ax.set_title("AudioSet-EV — filas por split y tarea binaria")
fig.tight_layout()
fig.savefig(FIG / "audioset_split_counts.png", bbox_inches="tight")
plt.show()
split_tbl


yt_id overlap (debe ser 0): {'train_val': 0, 'train_test': 0, 'val_test': 0}


/tmp/ipykernel_57290/2498447129.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,split,detection_label,n
0,balanced_train_ref,siren,124
1,test,siren,127
2,test,urban_negative,9953
3,train,siren,6010
4,train,urban_negative,9320
5,val,siren,1063
6,val,urban_negative,1643


## Protocolo de otro dominio (declarado, sin WAV)

LSSiren y UrbanSound8K no entran al train. Cuando existan WAV, todo LSSiren es `external_test_detection`; UrbanSound8K usa folds/`fsID`.


In [7]:
external_protocol = pd.DataFrame([
    {"corpus": "lssiren", "role": "external_test_detection", "split_rule": "corpus entero; nunca train", "wav_now": False},
    {"corpus": "urbansound8k", "role": "external_test_distractors", "split_rule": "folds oficiales + fsID", "wav_now": False},
    {"corpus": "sirennet", "role": "optional_clean_eval", "split_rule": "no mezclar en pool; test limpio opcional", "wav_now": True},
])
external_protocol.to_csv(TAB / "external_test_protocol.csv", index=False)
external_protocol


,corpus,role,split_rule,wav_now
0,lssiren,external_test_detection,corpus entero; nunca train,False
1,urbansound8k,external_test_distractors,folds oficiales + fsID,False
2,sirennet,optional_clean_eval,no mezclar en pool; test limpio opcional,True


## 4.2 Normalización y escalamiento

Peak/RMS por clip (sin estadística global). `StandardScaler` del vector tabular se ajusta **solo** en filas `split==train` de la muestra con audio.


In [8]:
def prepare_clip(path: str) -> AudioClip:
    clip = load_audio(path, sr=TARGET_SR, duration=AUDIOSET_DURATION_S)
    y = pad_or_trim(clip.y, clip.sr, AUDIOSET_DURATION_S)
    y = peak_normalize(y)
    return AudioClip(y=y, sr=clip.sr)

# Demostración: un clip train vs su versión peak-norm.
demo = splits[(splits["split"] == "train") & (splits["path"].map(lambda p: bool(p) and _P(str(p)).exists()))]
if demo.empty:
    print("Sin WAV de train en disco para la demo de normalización")
    demo_path = None
else:
    demo_path = demo.sample(1, random_state=SEED).iloc[0]["path"]
    raw = load_audio(demo_path, sr=TARGET_SR, duration=3.0)
    norm = peak_normalize(raw.y)
    fig, axes = plt.subplots(2, 1, figsize=(9, 4), sharex=True)
    t = np.arange(len(raw.y)) / raw.sr
    axes[0].plot(t, raw.y, lw=0.6)
    axes[0].set_ylabel("crudo")
    axes[1].plot(t, norm, lw=0.6, color="#3b6d9a")
    axes[1].set_ylabel("peak-norm")
    axes[1].set_xlabel("t (s)")
    fig.tight_layout()
    fig.savefig(FIG / "audioset_peak_norm_demo.png", bbox_inches="tight")
    plt.show()
    print(demo_path)


Sin WAV de train en disco para la demo de normalización


## 4.3 Transformaciones

Mono (ya lo es), resample 22 050 Hz, pad/trim a 10.0 s, log-mel `n_mels=64`, `n_fft=1024`, `hop=256`.


In [9]:
if demo_path is None:
    print("sin demo log-mel")
else:
    clip = prepare_clip(demo_path)
    mel = log_mel_spectrogram(clip, n_mels=64, n_fft=1024, hop=256)
    freqs, times, db = stft_db(clip)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    axes[0].pcolormesh(times, freqs, db, shading="auto", cmap="magma")
    axes[0].set_ylim(0, 8000)
    axes[0].set_title("STFT (dB)")
    axes[1].imshow(mel, origin="lower", aspect="auto", cmap="magma")
    axes[1].set_title(f"log-mel {mel.shape}")
    fig.tight_layout()
    fig.savefig(FIG / "audioset_transform_logmel.png", bbox_inches="tight")
    plt.show()
    print("log-mel shape", mel.shape, "sr", clip.sr, "n_samples", len(clip.y))


sin demo log-mel


## 4.4 Ingeniería de características

13 MFCC (media y std temporal) + ZCR, centroide, bandwidth, rolloff. Solo muestra con WAV; el extractor queda listo para el entrenamiento. Tipo: no aplanar multi-label.


In [10]:
MAX_PER_SPLIT_LABEL = 25
feat_source = splits[splits["path"].map(lambda p: bool(p) and _P(str(p)).exists())].copy()
parts = []
for (split, label), sub in feat_source.groupby(["split", "label"]):
    parts.append(sub.sample(min(MAX_PER_SPLIT_LABEL, len(sub)), random_state=SEED))
sample_df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

rows = []
for _, row in sample_df.iterrows():
    clip = prepare_clip(row["path"])
    mf = mfcc(clip, n_mfcc=13)
    rows.append({
        "path": row["path"],
        "yt_id": row["yt_id"],
        "split": row["split"],
        "label": row["label"],
        "detection_label": row["detection_label"],
        "type_eligible": bool(row["type_eligible"]),
        "zcr_mean": float(np.mean(zero_crossing_rate(clip))),
        "centroid_mean": float(np.mean(spectral_centroid(clip))),
        "bandwidth_mean": float(np.mean(spectral_bandwidth(clip))),
        "rolloff_mean": float(np.mean(spectral_rolloff(clip))),
        **{f"mfcc_{i+1}_mean": float(np.mean(mf[i])) for i in range(min(13, mf.shape[0]))},
        **{f"mfcc_{i+1}_std": float(np.std(mf[i])) for i in range(min(13, mf.shape[0]))},
    })
feat = pd.DataFrame(rows)
if feat.empty:
    print("Sin WAV suficientes para features")
    scaled = feat
else:
    num_cols = [c for c in feat.columns if c.startswith("mfcc_") or c.endswith("_mean")]
    scaler = StandardScaler()
    train_mask = feat["split"] == "train"
    if train_mask.any():
        scaler.fit(feat.loc[train_mask, num_cols])
        scaled_vals = scaler.transform(feat[num_cols])
    else:
        print("La muestra no tiene split=train; no se ajusta scaler (evita filtrar val/test).")
        scaled_vals = np.zeros((len(feat), len(num_cols)))
    scaled = feat.copy()
    for i, col in enumerate(num_cols):
        scaled[f"{col}_scaled"] = scaled_vals[:, i]
    feat.to_csv(TAB / "audioset_ev_feature_sample.csv", index=False)
    scaled.to_csv(TAB / "audioset_ev_feature_sample_scaled.csv", index=False)
    print("scaler n_train", int(train_mask.sum()), "n_features", len(num_cols))
feat.head() if not feat.empty else feat


Sin WAV suficientes para features


""


## Resultados

- Limpieza = índice y flags, no un WAV nuevo.
- Splits oficiales + val agrupado; overlap de `yt_id` a cero.
- Scaler solo en train. LSSiren/UrbanSound8K declarados como otro dominio.
- No se entrenó modelo.


In [11]:
result = {
    "seed": SEED,
    "n_index": int(len(clean)),
    "n_keep": int(clean["keep"].sum()),
    "overlap": overlap if "overlap" in globals() else None,
    "n_feature_rows": int(len(feat)) if "feat" in globals() else 0,
    "figures": sorted(p.name for p in FIG.glob("audioset_*.png")),
}
result


{'seed': 7,
 'n_index': 28240,
 'n_keep': 28240,
 'overlap': {'train_val': 0, 'train_test': 0, 'val_test': 0},
 'n_feature_rows': 0,
 'figures': ['audioset_class_balance.png', 'audioset_split_counts.png']}